# 01 — Data Understanding

**Objective:** Understand the raw Delhi AQI dataset before any cleaning happens — its shape, types, quality issues, and coverage — per Handbook Section 10.2 (notebook structure) and D.7 (Dataset Understanding Framework).

**Business context:** This dataset is the foundation for forecasting future AQI values (PRD Business Objective BO-002). Every downstream cleaning/modeling decision depends on correctly understanding what's actually in it — not what we assume is in it.

**Dataset:** `data/raw/Delhi_AQI_Dataset.csv` — see `data/metadata/DATASET_SOURCE.md` for full provenance, license (CC BY 4.0), and a documented caveat about how pollutant columns were derived.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing.data_loader import load_dataset
from src.preprocessing.data_validator import validate_dataset
from config.paths import RAW_DATA_DIR
from config.constants import REQUIRED_DATASET_COLUMNS

plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", None)

## Load the dataset

Using our own `load_dataset()` (FR-DATA-001). Dates are `DD/MM/YY` in the source file — confirmed by inspecting rows where the first number exceeds 12 (e.g. `29/12/24`), and verified against the loader's default parsing, which silently mis-parsed 36% of rows before this was fixed (see `CHANGELOG.md`).

In [2]:
df = load_dataset(RAW_DATA_DIR / "Delhi_AQI_Dataset.csv", date_format="%d/%m/%y")
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
df.head()

2026-08-18 00:25:19,423 | INFO | src.preprocessing.data_loader | Loaded dataset 'Delhi_AQI_Dataset.csv': 2191 rows, 11 columns.


Shape: (2191, 11)
Date range: 2018-01-01 to 2024-12-31


,City,Date,AQI,PM2.5,PM10,NO2,SO2,CO,O3,Unnamed: 9,Unnamed: 10
0,Delhi,2018-01-01,406,223.3,438.48,336.98,462.84,4.26,385.7,NaN,NaN
1,Delhi,2018-01-02,418,229.9,451.44,346.94,476.52,4.39,397.1,NaN,NaN
2,Delhi,2018-01-03,382,210.1,412.56,317.06,435.48,4.01,362.9,NaN,NaN
3,Delhi,2018-01-04,366,201.3,395.28,303.78,417.24,3.84,347.7,NaN,NaN
4,Delhi,2018-01-05,390,214.5,421.20,323.70,444.60,4.10,370.5,NaN,NaN


**Observation:** the file loads with 11 columns, not the 9 documented in `DATASET_SOURCE.md`. Let's see why.

In [3]:
print(list(df.columns))
df[["Unnamed: 9", "Unnamed: 10"]].isna().mean()

['City', 'Date', 'AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Unnamed: 9', 'Unnamed: 10']


Unnamed: 9     1.0
Unnamed: 10    1.0
dtype: float64

**Finding:** the source CSV's header has two trailing commas (`...,O3,,`), which pandas reads as two fully-empty columns. This is a genuine data-quality issue, not a bug in our loader — it will be dropped explicitly in `02_data_cleaning.ipynb` (not silently, so the action is auditable).

## Structural validation (FR-DATA-002)

In [4]:
report = validate_dataset(df, REQUIRED_DATASET_COLUMNS)
print(report.summary())

2026-08-18 00:25:19,860 | INFO | src.preprocessing.data_validator | Dataset validation passed.
Validation status: VALID
Shape: 2191 rows x 11 columns
Duplicate rows: 0
Columns with missing values (%): {'Unnamed: 9': 100.0, 'Unnamed: 10': 100.0}


Validation status: VALID
Shape: 2191 rows x 11 columns
Duplicate rows: 0
Columns with missing values (%): {'Unnamed: 9': 100.0, 'Unnamed: 10': 100.0}


**Interpretation:** validation passes — all 9 mandatory columns (ML-DATA-002) are present. The dataset has **zero missing values** in any real data column (only the two junk columns are 100% empty, which is expected and handled in cleaning) and **zero duplicate rows**. This is an unusually clean source for a real-world dataset — worth double-checking rather than taking at face value, which the next cells do.

## Data types

In [5]:
df.dtypes

City                   object
Date           datetime64[ns]
AQI                     int64
PM2.5                 float64
PM10                  float64
NO2                   float64
SO2                   float64
CO                    float64
O3                    float64
Unnamed: 9            float64
Unnamed: 10           float64
dtype: object

**Interpretation:** `AQI` is integer, all pollutant columns are float, `Date` correctly parsed as `datetime64`, `City` is string. No type coercion issues — nothing here needs correction before analysis.

## Descriptive statistics

In [6]:
df.describe()

,Date,AQI,PM2.5,PM10,NO2,SO2,CO,O3,Unnamed: 9,Unnamed: 10
count,2191,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,2191.000000,0.0,0.0
mean,2021-05-02 10:01:22.154267392,208.285714,114.557143,224.948571,172.877143,237.445714,2.187307,197.871429,NaN,NaN
min,2018-01-01 00:00:00,41.000000,22.550000,44.280000,34.030000,46.740000,0.430000,38.950000,NaN,NaN
25%,2019-07-02 12:00:00,117.000000,64.350000,126.360000,97.110000,133.380000,1.230000,111.150000,NaN,NaN
50%,2021-01-01 00:00:00,190.000000,104.500000,205.200000,157.700000,216.600000,2.000000,180.500000,NaN,NaN
75%,2023-07-02 12:00:00,289.000000,158.950000,312.120000,239.870000,329.460000,3.030000,274.550000,NaN,NaN
max,2024-12-31 00:00:00,494.000000,271.700000,533.520000,410.020000,563.160000,5.190000,469.300000,NaN,NaN
std,NaN,106.614654,58.638060,115.143827,88.490163,121.540706,1.119494,101.283922,NaN,NaN


**Interpretation:**
- AQI ranges from 41 (Good/Moderate) to 494 (Hazardous) — the full severity spectrum is represented, which is good for a forecasting model (no truncated range).
- Mean AQI is ~208 ("Unhealthy" per our `config.constants.AQI_CATEGORIES` breakpoints), consistent with Delhi's well-documented pollution levels.
- Standard deviation (~107) is large relative to the mean — substantial day-to-day variability, which is exactly what a forecasting model needs to be useful (a dataset with near-constant AQI wouldn't be interesting to forecast).

## Coverage: is every calendar day present?

In [7]:
full_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
missing_dates = full_range.difference(df["Date"])
print(f"Expected calendar days in range: {len(full_range)}")
print(f"Actual rows: {len(df)}")
print(f"Missing calendar days: {len(missing_dates)} ({len(missing_dates)/len(full_range)*100:.1f}%)")
print()
print("First 10 missing dates:")
print(missing_dates[:10].tolist())

Expected calendar days in range: 2557
Actual rows: 2191
Missing calendar days: 366 (14.3%)

First 10 missing dates:
[Timestamp('2020-02-29 00:00:00'), Timestamp('2022-01-01 00:00:00'), Timestamp('2022-01-02 00:00:00'), Timestamp('2022-01-03 00:00:00'), Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00'), Timestamp('2022-01-06 00:00:00'), Timestamp('2022-01-07 00:00:00'), Timestamp('2022-01-08 00:00:00'), Timestamp('2022-01-09 00:00:00')]


**Finding — important for later milestones:** 366 of 2,557 calendar days (14.3%) between 2018-01-01 and 2024-12-31 are simply absent from the file, not present-with-NaN. This is **not a "missing value"** the validator would catch (there's no row to have a missing value in) — it's a gap in time-series coverage. This matters directly for **Milestone 3 (Feature Engineering)**: lag features like `AQI(t-1)` and rolling windows must account for these gaps (e.g. by reindexing to a full daily calendar and explicitly deciding how to treat gap-days) rather than assuming row *N-1* is always "yesterday."

## Single-city confirmation

In [8]:
print(df["City"].unique())

['Delhi']


**Interpretation:** confirms single-city scope (Delhi only), consistent with PRD's V1 scope (multi-city forecasting is explicitly out-of-scope / a V2.0 feature).

## Conclusions

1. Dataset loads cleanly with our pipeline once the `DD/MM/YY` date format is handled explicitly.
2. Two fully-empty junk columns need dropping (cleaning step).
3. Zero missing values in real columns, zero duplicate rows — validation passes outright.
4. **366 calendar-day gaps (14.3%)** exist and must be handled explicitly during feature engineering, not ignored.
5. AQI has good range and variance for forecasting purposes.

## Next steps
→ `02_data_cleaning.ipynb`: drop the junk columns, formally re-run duplicate/outlier detection using `src/preprocessing/data_cleaner.py`, and persist a clean dataset to `data/processed/`.